In [2]:
import os
import numpy as np, pandas as pd

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
ROOT = f"{COMP}/train_series"
ser = pd.read_csv(f"{COMP}/train_series.csv")

def n_slices(study, series):
    p = os.path.join(ROOT, study, series)
    return len([f for f in os.listdir(p) if f.endswith(".dcm")]) if os.path.isdir(p) else 0

ser["n"] = [n_slices(s, r) for s, r in zip(ser["StudyInstanceUID"], ser["SeriesInstanceUID"])]
ser = ser[ser["n"] > 0]

ser["score"] = ser["Fluid_Sensitive"] * 2 + ser["Fat_Suppression"]
chosen = (ser.sort_values(["score", "n"], ascending=False)
             .groupby(["StudyInstanceUID", "Anatomical_Plane"], as_index=False).head(1))

def ringkas(s):
    return pd.Series({"jumlah seri": len(s), "rata-rata": round(s.mean(), 1), "median": s.median(),
                      "min": s.min(), "P10": s.quantile(.10), "P25": s.quantile(.25),
                      "P75": s.quantile(.75), "P90": s.quantile(.90), "maks": s.max(),
                      "% < 16 irisan": round((s < 16).mean() * 100, 1),
                      "% <= 32 irisan": round((s <= 32).mean() * 100, 1)})

print("=== SEMUA SERI ===")
print(ser.groupby("Anatomical_Plane")["n"].apply(ringkas).unstack().to_string())
print("\n=== SERI YANG TERPILIH (satu per bidang per studi) ===")
print(chosen.groupby("Anatomical_Plane")["n"].apply(ringkas).unstack().to_string())
print("\nGabungan seri terpilih:")
print(ringkas(chosen["n"]).to_string())
print("\nJumlah studi:", ser["StudyInstanceUID"].nunique())

=== SEMUA SERI ===
                  jumlah seri  rata-rata  median   min   P10   P25   P75   P90   maks  % < 16 irisan  % <= 32 irisan
Anatomical_Plane                                                                                                    
Axial                  5898.0       39.8    32.0  12.0  22.0  28.0  38.0  44.0  254.0            1.1            53.2
Coronal                8609.0       28.3    30.0  11.0  18.0  25.0  32.0  36.0   80.0            1.4            80.9
Sagittal               9864.0       34.6    30.0  11.0  21.0  25.0  33.0  36.0  320.0            1.0            70.3

=== SERI YANG TERPILIH (satu per bidang per studi) ===
                  jumlah seri  rata-rata  median   min   P10   P25   P75   P90   maks  % < 16 irisan  % <= 32 irisan
Anatomical_Plane                                                                                                    
Axial                  4407.0       40.4    32.0  14.0  22.0  28.0  38.0  44.0  192.0            0.1      

In [3]:
import glob, pydicom
def geom(study, series):
    f = sorted(glob.glob(os.path.join(ROOT, study, series, "*.dcm")))[:1]
    if not f: return np.nan, np.nan
    d = pydicom.dcmread(f[0], stop_before_pixels=True)
    try: return float(d.PixelSpacing[0]), float(d.SliceThickness)
    except Exception: return np.nan, np.nan

g = [geom(s, r) for s, r in zip(chosen["StudyInstanceUID"], chosen["SeriesInstanceUID"])]
chosen["piksel_mm"], chosen["tebal_mm"] = zip(*g)
chosen["rasio"] = chosen["tebal_mm"] / chosen["piksel_mm"]
print(chosen.groupby("Anatomical_Plane")[["piksel_mm", "tebal_mm", "rasio"]].median().round(2))

                  piksel_mm  tebal_mm  rasio
Anatomical_Plane                            
Axial                  0.32       3.0  10.50
Coronal                0.32       3.0  10.24
Sagittal               0.33       3.0   9.60
